# 2.1 · JSON-RPC messages, _meta and error codes

**JSON-RPC 消息、_meta 与错误码**

> **After this lesson you can:** Read and write any MCP message by hand, use the `_meta` envelope and its naming rules correctly, and diagnose a failure from its JSON-RPC error code.

Official reading: [Spec: Base protocol overview](https://modelcontextprotocol.io/specification/2026-07-28/basic/index) · [Schema reference](https://modelcontextprotocol.io/specification/2026-07-28/schema)

Run the cells top to bottom (`Shift+Enter`), then change things and re-run. Everything runs locally: servers live in this kernel, in temp folders, or on localhost. Cells that call a real LLM are skipped unless `ANTHROPIC_API_KEY` is set.

### 1. A message classifier you can trust
Before talking to a server, encode the rules from the lesson as code. `inspect_message` classifies a decoded JSON object and lists every rule it breaks. We will reuse it on the server's real replies.

In [ ]:
import json


def show(obj, limit=900):
    text = json.dumps(obj, indent=2, ensure_ascii=False)
    print(text if len(text) <= limit else text[:limit] + "\n  ...")


REQUIRED_ENVELOPE = ("io.modelcontextprotocol/protocolVersion", "io.modelcontextprotocol/clientCapabilities")


def inspect_message(msg: dict) -> tuple[str, list[str]]:
    """Return (kind, problems) for one decoded MCP / JSON-RPC message."""
    problems = []
    if msg.get("jsonrpc") != "2.0":
        problems.append('jsonrpc must be exactly "2.0"')
    has_id = "id" in msg
    if "method" in msg:
        kind = "request" if has_id else "notification"
        if has_id:
            rid = msg["id"]
            if rid is None:
                problems.append("request id MUST NOT be null (MCP rule, stricter than JSON-RPC)")
            elif isinstance(rid, bool) or not isinstance(rid, (str, int)):
                problems.append("request id must be a string or an integer")
            meta = (msg.get("params") or {}).get("_meta") or {}
            missing = [k for k in REQUIRED_ENVELOPE if k not in meta]
            if missing:
                problems.append(f"request envelope is missing {missing} -> server answers -32602")
    elif "result" in msg or "error" in msg:
        kind = "response"
        if "result" in msg and "error" in msg:
            problems.append("a response carries result OR error, never both")
        if "result" in msg:
            if msg.get("id") is None:
                problems.append("a result response must echo the request id")
            if "resultType" not in msg["result"]:
                problems.append("result lacks resultType (a client treats it as 'complete')")
        if "error" in msg and not isinstance(msg["error"].get("code"), int):
            problems.append("error.code must be an integer")
    else:
        kind = "invalid"
        problems.append("neither method nor result/error present")
    return kind, problems


ENVELOPE = {"io.modelcontextprotocol/protocolVersion": "2026-07-28",
            "io.modelcontextprotocol/clientCapabilities": {},
            "io.modelcontextprotocol/clientInfo": {"name": "notebook", "version": "0.1"}}

samples = {
    "good request": {"jsonrpc": "2.0", "id": 1, "method": "tools/list", "params": {"_meta": ENVELOPE}},
    "null id": {"jsonrpc": "2.0", "id": None, "method": "tools/list", "params": {"_meta": ENVELOPE}},
    "no envelope": {"jsonrpc": "2.0", "id": "a", "method": "tools/list"},
    "notification": {"jsonrpc": "2.0", "method": "notifications/progress",
                     "params": {"progressToken": "t", "progress": 1}},
    "old-style result": {"jsonrpc": "2.0", "id": 1, "result": {"tools": []}},
    "error": {"jsonrpc": "2.0", "id": 1, "error": {"code": -32601, "message": "Method not found"}},
}
for label, msg in samples.items():
    kind, problems = inspect_message(msg)
    print(f"{label:<17} -> {kind:<12} {problems or 'OK'}")

### 2. `_meta` key rules and the reserved prefixes
Implement the key grammar from the spec: an optional reverse-DNS prefix ending in `/`, then a name. A prefix is **reserved** when its *second* label is `modelcontextprotocol` or `mcp`.

In [ ]:
import re

LABEL = r"[A-Za-z](?:[A-Za-z0-9-]*[A-Za-z0-9])?"
NAME = r"(?:[A-Za-z0-9](?:[A-Za-z0-9._-]*[A-Za-z0-9])?)?"
KEY = re.compile(rf"^(?:(?P<prefix>{LABEL}(?:\.{LABEL})*)/)?(?P<name>{NAME})$")
OTEL_KEYS = {"traceparent", "tracestate", "baggage"}


def check_meta_key(key: str) -> str:
    m = KEY.match(key)
    if not m:
        return "INVALID"
    prefix = m.group("prefix")
    if prefix is None:
        return "OpenTelemetry exception (reserved, no prefix)" if key in OTEL_KEYS else "valid, unprefixed"
    labels = prefix.split(".")
    if len(labels) > 1 and labels[1] in ("modelcontextprotocol", "mcp"):
        return "RESERVED for MCP"
    return "valid, vendor prefix"


for key in ["io.modelcontextprotocol/protocolVersion", "dev.mcp/anything", "org.modelcontextprotocol.api/x",
            "com.mcp.tools/y", "com.example.mcp/tenant", "com.acme/tenant", "traceparent", "progressToken",
            "1bad.com/x", "com.acme/-name", "com.acme-/x"]:
    print(f"{key:<42} {check_meta_key(key)}")

Notice `com.example.mcp/tenant` is **not** reserved (its second label is `example`), while `com.mcp.tools/y` **is**.

### 3. Where does an error code come from?
Codify the allocation policy so you can read any code at a glance.

In [ ]:
MCP_DEFINED = {-32020: "HeaderMismatch", -32021: "MissingRequiredClientCapability",
               -32022: "UnsupportedProtocolVersion"}
JSONRPC_STANDARD = {-32700: "Parse error", -32600: "Invalid request", -32601: "Method not found",
                    -32602: "Invalid params", -32603: "Internal error"}


def explain_code(code: int) -> str:
    if code in JSONRPC_STANDARD:
        return f"JSON-RPC standard: {JSONRPC_STANDARD[code]}"
    if code in MCP_DEFINED:
        return f"MCP spec: {MCP_DEFINED[code]}"
    if -32019 <= code <= -32000:
        extra = " (old 'resource not found' - accept from legacy servers)" if code == -32002 else ""
        return "legacy implementation-defined range: assume nothing" + extra
    if -32099 <= code <= -32020:
        return "reserved for the MCP spec but undefined: a server MUST NOT emit it"
    if -32768 <= code <= -32000:
        return "JSON-RPC reserved range: do not use for app errors"
    return "application-defined (outside the JSON-RPC reserved range)"


for code in [-32700, -32602, -32020, -32021, -32022, -32002, -32001, -32042, -32050, -32500, -1, 4004]:
    print(f"{code:>7}  {explain_code(code)}")

### 4. Provoke every error from a real server over stdio
Now a real SDK server runs as a subprocess and we talk to it with raw JSON lines. One tool, `delete`, needs the user's confirmation. It declares that through a resolver that elicits, so the server needs the client's `elicitation` capability.

In [ ]:
import queue
import subprocess
import sys
import tempfile
import textwrap
import threading
from pathlib import Path

workdir = Path(tempfile.mkdtemp())
(workdir / "lab_server.py").write_text(textwrap.dedent('''
    from typing import Annotated
    from pydantic import BaseModel
    from mcp.server import MCPServer
    from mcp.server.mcpserver import Elicit, Resolve

    mcp = MCPServer("error-lab", version="1.0.0")

    @mcp.tool()
    def add(a: int, b: int) -> int:
        """Add two integers."""
        return a + b

    @mcp.tool()
    def divide(a: float, b: float) -> float:
        """Divide a by b."""
        if b == 0:
            raise ValueError("division by zero")
        return a / b

    class Confirm(BaseModel):
        ok: bool

    def ask(target: str):
        return Elicit(f"Really delete {target}?", Confirm)

    @mcp.tool()
    def delete(target: str, confirm: Annotated[Confirm, Resolve(ask)]) -> str:
        """Delete something after the user confirms."""
        return f"deleted {target}" if confirm.ok else "kept"

    if __name__ == "__main__":
        mcp.run(transport="stdio")
'''))

proc = subprocess.Popen([sys.executable, str(workdir / "lab_server.py")], stdin=subprocess.PIPE,
                        stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True)
inbox = queue.Queue()
threading.Thread(target=lambda: [inbox.put(line) for line in proc.stdout], daemon=True).start()


def send_raw(text: str, wait: float = 5.0):
    """Write one line to the server's stdin and return the next line it writes (or None)."""
    proc.stdin.write(text + "\n")
    proc.stdin.flush()
    try:
        return json.loads(inbox.get(timeout=wait))
    except queue.Empty:
        return None


def rpc(id_, method, params=None, *, meta=ENVELOPE):
    params = dict(params or {})
    if meta is not None:
        params["_meta"] = meta
    return send_raw(json.dumps({"jsonrpc": "2.0", "id": id_, "method": method, "params": params}))


reply = rpc(1, "server/discover")
show(reply)
print("classifier:", inspect_message(reply))

Now break things one at a time and run each reply through both helpers:

In [ ]:
def report(label, reply):
    if reply is None:
        print(f"{label:<34} -> (no reply)")
    elif "error" in reply:
        err = reply["error"]
        print(f"{label:<34} -> {err['code']} {explain_code(err['code'])}\n{'':<38}message: {err['message'][:90]}")
        if "data" in err:
            print(f"{'':<38}data: {json.dumps(err['data'])}")
    else:
        r = reply["result"]
        print(f"{label:<34} -> result resultType={r.get('resultType')!r} isError={r.get('isError')}")


report("unknown method", rpc(2, "tasks/list"))
report("missing _meta envelope", rpc(3, "tools/list", meta=None))
report("unsupported protocol version", rpc(4, "tools/list",
       meta={**ENVELOPE, "io.modelcontextprotocol/protocolVersion": "1900-01-01"}))
report("needs undeclared elicitation", rpc(5, "tools/call", {"name": "delete", "arguments": {"target": "prod"}}))
report("tool raises (NOT a protocol error)", rpc(6, "tools/call", {"name": "divide", "arguments": {"a": 1, "b": 0}}))

Read the last two carefully. Calling `delete` without declaring `elicitation` gives **-32021**, and `data.requiredCapabilities` tells the client exactly what to add. A tool that raises is **not** a JSON-RPC error: it is a `complete` result with `isError: true`, which the host shows to the model.

What happens if the client *does* declare elicitation? The server can no longer send its own `elicitation/create` request (servers never initiate requests in 2026-07-28), so it answers with `resultType: "input_required"`:

In [ ]:
reply = rpc(7, "tools/call", {"name": "delete", "arguments": {"target": "prod"}},
            meta={**ENVELOPE, "io.modelcontextprotocol/clientCapabilities": {"elicitation": {}}})
result = reply["result"]
print("resultType:", result["resultType"])
show(result["inputRequests"])
print("requestState (opaque, echoed back on retry):", result["requestState"][:40], "...")

Lesson 5.3 finishes this round trip. For now the point is that `resultType` tells the client how to parse `result`.

Finally, malformed JSON-RPC. The spec says a parse error is answered with `id: null`, but this SDK's stdio server simply drops lines it cannot parse or that break the JSON-RPC shape. **Never rely on getting an answer to garbage.** Always use client-side timeouts.

In [ ]:
for label, line in [("not JSON at all", '{"jsonrpc": "2.0", "id": 8, "method": "tools/l'),
                    ("null id", json.dumps({"jsonrpc": "2.0", "id": None, "method": "tools/list",
                                            "params": {"_meta": ENVELOPE}})),
                    ("jsonrpc 1.0", json.dumps({"jsonrpc": "1.0", "id": 9, "method": "tools/list",
                                                "params": {"_meta": ENVELOPE}}))]:
    report(label, send_raw(line, wait=1.5))

report("server still healthy?", rpc(10, "tools/call", {"name": "add", "arguments": {"a": 2, "b": 40}}))
proc.stdin.close()
exit_code = proc.wait(timeout=10)   # closing stdin is the graceful shutdown signal

### 5. The same errors over Streamable HTTP, with status codes
On HTTP each error also gets an HTTP status: `400` for envelope, header and version problems and `404` for an unknown method. Here the server runs in a background thread on `127.0.0.1`.

In [ ]:
import logging
import socket
import time

import httpx
import uvicorn
from mcp.server import MCPServer

for noisy in ("httpx", "mcp", "uvicorn"):
    logging.getLogger(noisy).setLevel(logging.ERROR)

http_server = MCPServer("error-lab-http")


@http_server.tool()
def add(a: int, b: int) -> int:
    """Add two integers."""
    return a + b


sock = socket.socket()
sock.bind(("127.0.0.1", 0))
port = sock.getsockname()[1]
sock.close()
uv = uvicorn.Server(uvicorn.Config(http_server.streamable_http_app(), host="127.0.0.1", port=port,
                                   log_level="warning"))
threading.Thread(target=uv.run, daemon=True).start()
while not uv.started:
    time.sleep(0.05)
URL = f"http://127.0.0.1:{port}/mcp"
web = httpx.Client(trust_env=False, timeout=10)
BASE_HEADERS = {"Accept": "application/json, text/event-stream", "Content-Type": "application/json",
                "MCP-Protocol-Version": "2026-07-28"}


def post(label, body, headers):
    r = web.post(URL, content=body if isinstance(body, str) else json.dumps(body),
                 headers={**BASE_HEADERS, **headers})
    data = r.json()
    code = data.get("error", {}).get("code")
    summary = f"error {code}: {data['error']['message'][:70]}" if code else f"result {data['result'].get('structuredContent')}"
    print(f"{label:<28} HTTP {r.status_code}  {summary}")


call = {"jsonrpc": "2.0", "id": 1, "method": "tools/call",
        "params": {"name": "add", "arguments": {"a": 2, "b": 3}, "_meta": ENVELOPE}}
post("well-formed", call, {"Mcp-Method": "tools/call", "Mcp-Name": "add"})
post("Mcp-Name disagrees", call, {"Mcp-Method": "tools/call", "Mcp-Name": "sub"})
post("unknown method", {"jsonrpc": "2.0", "id": 2, "method": "nope/x", "params": {"_meta": ENVELOPE}},
     {"Mcp-Method": "nope/x"})
post("unsupported version", {**call, "params": {**call["params"], "_meta": {
    **ENVELOPE, "io.modelcontextprotocol/protocolVersion": "1900-01-01"}}},
     {"Mcp-Method": "tools/call", "Mcp-Name": "add", "MCP-Protocol-Version": "1900-01-01"})
post("missing envelope", {"jsonrpc": "2.0", "id": 3, "method": "tools/list", "params": {}},
     {"Mcp-Method": "tools/list"})
post("not JSON", '{"jsonrpc": "2.0", "id": 4,', {"Mcp-Method": "tools/list"})

Over HTTP the parse error *is* answered, as `-32700` with `id: null`, because the server could not read the id. Lesson 2.3 digs into the headers.

### 6. `_meta` passthrough: trace context and vendor keys
`Client.call_tool(..., meta=...)` merges your keys into the envelope the SDK builds. This tool echoes what arrived so you can see a W3C `traceparent` and a vendor-prefixed tenant key travelling next to the protocol fields.

In [ ]:
from mcp import Client
from mcp.server.mcpserver import Context

echo = MCPServer("meta-echo")


@echo.tool()
def show_meta(ctx: Context) -> str:
    """Return the request's _meta exactly as the server received it."""
    return json.dumps(dict(ctx.request_context.meta or {}), indent=2, default=str)


async with Client(echo) as client:
    result = await client.call_tool("show_meta", {}, meta={
        "traceparent": "00-0af7651916cd43dd8448eb211c80319c-00f067aa0ba902b7-01",
        "com.acme/tenant": "tenant-42",
    })
    print(result.content[0].text)
    print("\nserverInfo in the result's _meta:", result.meta["io.modelcontextprotocol/serverInfo"])

The SDK filled in `protocolVersion`, `clientCapabilities` and `clientInfo` for you. Server-side, the SDK's OpenTelemetry middleware reads `traceparent` from this same `_meta` to parent its span.

### 7. Icons, and checking them like an untrusted input
Servers can attach `icons` to their implementation info and to tools. A client must vet them before rendering anything.

In [ ]:
from mcp_types import Icon

PNG_DOT = "data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAYAAAAfFcSJAAAADUlEQVR42mP8z8BQDwAEhQGAhKmMIQAAAABJRU5ErkJggg=="
iconic = MCPServer("iconic", icons=[Icon(src=PNG_DOT, mime_type="image/png", sizes=["1x1"])])


@iconic.tool(icons=[Icon(src="javascript:alert(1)"), Icon(src="https://iconic.example/calc.svg",
                                                            mime_type="image/svg+xml", sizes=["any"])])
def calc(expression: str) -> str:
    """Pretend calculator."""
    return expression


def icon_verdict(icon) -> str:
    scheme = icon.src.split(":", 1)[0].lower()
    if scheme not in ("https", "data"):
        return f"REJECT unsafe scheme {scheme!r}"
    if icon.mime_type == "image/svg+xml":
        return "allow only after sanitising the SVG (it can embed script)"
    return "ok: fetch without credentials, then verify magic bytes"


async with Client(iconic) as client:
    print("server icons:", [icon_verdict(i) for i in client.server_info.icons or []])
    for tool in (await client.list_tools()).tools:
        for icon in tool.icons or []:
            print(f"tool {tool.name}: {icon.src[:40]:<40} -> {icon_verdict(icon)}")

### Your turn
1. Extend `inspect_message` so it flags a request whose `_meta` contains a key that fails `check_meta_key`, and a notification that carries an `id: null`. Re-run it on a few hand-made messages.
2. In section 4, call `delete` again but declare `{"elicitation": {"url": {}}}` only. Does the server accept URL-only elicitation for a form question? Read `data.requiredCapabilities` to find out.
3. In section 5, send a well-formed request with `MCP-Protocol-Version: 2026-07-28` in the header but `"2025-11-25"` in the body `_meta`. Which code wins, `-32020` or `-32022`, and why does checking the header first make sense?
4. Using `explain_code`, decide which code your own server should use for "customer not found" inside a tool. (Trick question: should it be a JSON-RPC error at all?)